<h1>Mindestanforderungen 4</h1>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from currency_converter import CurrencyConverter
c = CurrencyConverter()
from datetime import date

In [ ]:
df = pd.read_csv("stack-overflow-developer-survey-2025/survey_results_public_without_SO.csv")

df.head()

In [ ]:
# Alter in numerischen Wert umwandeln

age_map = {
    'Under 18 years old': 17,
    '18-24 years old': 21,
    '25-34 years old': 29,
    '35-44 years old': 39,
    '45-54 years old': 49,
    '55-64 years old': 59,
    '65 years or older': 70
}

df['AgeNum'] = df['Age'].map(age_map)

#print (df['AgeNum'].describe())


#df[df['AgeNum'] == 70]
# Alle Ü65 rausschmeißen
df = df[df['AgeNum'] <= 65]


Erstellen Sie mindestens 2 neue Features

In [ ]:
# Mit Currency Converter Jahresgehalt in USD umwandeln

# Currency Spalte alles nach den ersten 3 Buchstaben abschneiden
df['Currency'] = df['Currency'].str[:3]

df = df[df['Currency'] != 'HRK']

#print (df['Currency'].unique())

# Spalte CompTotal in USD umwandeln und in Spalte convertedCompTotal speichern

def convert_to_usd(currency, comp):
    if currency not in c.currencies:
        return np.nan
    if currency == "RUB":
        converted = c.convert(comp, currency, 'USD', date=date(2022, 3, 1))
    elif currency == "HRK":
        converted = c.convert(comp, currency, 'USD', date=date(2022, 12, 30))
    else:
        converted = c.convert(comp, currency, 'USD', date=date(2025, 10, 6))
    return converted

df['ConvertedCompTotal'] = df.apply(lambda row: convert_to_usd(row['Currency'], row['CompTotal']), axis=1)

df.to_csv("survey_with_converted_comp.csv", index=False)



In [ ]:
q95 = (df[df['ConvertedCompTotal'] <= df['ConvertedCompTotal'].quantile(0.95)])

In [ ]:
q95['ConvertedCompTotal'].describe()

Untersuchen Sie die Korrelationen gesamthaft und graphisch für mindestens 2 Feature-Kombinationen

In [ ]:
# Bei Moritz zu finden

Erstellen / Konvertieren Sie eine Time Series

In [ ]:
# Wechselkurse analysieren
# Line Chart USD von jedem Tag

start = "2025-06-30"
end = "2025-11-03"

dates = pd.bdate_range(start, end)
print (dates)

def get_rates(date, curr):
    cc = c.convert(1, curr, 'USD', date=date)
    return cc

rates = []

for d in dates:
    rate = get_rates(d, 'EUR')
    rates.append(rate)

frame = pd.DataFrame({'date': dates})
frame.set_index('date', inplace=True)

frame['EUR'] = rates

rates = []

for d in dates:
    rate = get_rates(d, 'GBP')
    rates.append(rate)

frame['GBP'] = rates

rates = []

for d in dates:
    rate = get_rates(d, 'INR')
    rates.append(rate)

frame['INR'] = rates

rates = []

for d in dates:
    rate = get_rates(d, 'CAD')
    rates.append(rate)

frame['CAD'] = rates

frame

    

In [ ]:
fig, ax = plt.subplots(2,2,figsize=(20, 16))

ax[0,0].set_title('EUR to USD Exchange Rate')
ax[0,0].plot(frame.index, frame['EUR'])

ax[0,1].set_title('INR to USD Exchange Rate')
ax[0,1].plot(frame.index, frame['INR'])

ax[1,0].set_title('GBP to USD Exchange Rate')
ax[1,0].plot(frame.index, frame['GBP'])

ax[1,1].set_title('CAD to USD Exchange Rate')
ax[1,1].plot(frame.index, frame['CAD'])

plt.show()


Erstellen Sie eine Analyse basierend auf den Zeitdaten

In [ ]:

start = "2025-06-30"
end = "2025-11-03"

dates = pd.bdate_range(start, end)

def get_rates(date, curr):
    cc = c.convert(1, curr, 'USD', date=date)
    return cc

frame1 = pd.DataFrame({'date': dates}).set_index('date')

for curr in ['EUR', 'GBP', 'INR', 'CAD']:
    rates = []
    for d in dates:
        rates.append(get_rates(d, curr))
    frame1[curr] = rates

# Tag mit minimaler lokaler Volatilität

# tägliche prozentuale Änderungen (Returns)
returns = frame1[['EUR', 'GBP', 'INR', 'CAD']].pct_change()

# rollierende Standardabweichung über 7 Tage
window = 7  # ca. ±3 Handelstage
rolling_vol = returns.rolling(window=window, center=True).std()

# gemeinsamen Volatilitäts-Score pro Tag bilden
vol_score = rolling_vol.mean(axis=1)

# Tage ohne vollständige Daten entfernen
vol_score = vol_score.dropna()

# Tag mit minimaler Volatilität bestimmen
best_date = vol_score.idxmin()
best_score = vol_score.loc[best_date]

print("Empfohlener Stichtag:", best_date.date())
print("Volatilitäts-Score an diesem Tag:", best_score)
